# 05 Create SFINCS Scenarios

## Austin inland adaptation

Create scenario folders for the shared Austin event catalog. Each scenario will combine Wflow-routed discharge, direct rainfall, antecedent state, and the reviewed SFINCS domain set once handoff outlets are approved.


## Stage Contract

Requires:
- event catalog and stress/training set
- Wflow handoff manifest
- SFINCS base model domains

Produces:
- event-specific SFINCS scenario folders
- forcing manifests for Wflow discharge and direct rainfall
- cluster launch plan

Next:
- evaluate completed flood outputs in `06_evaluate.ipynb`.


## Imports and Runtime


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parent
location_name = location_root.name
repo_root = location_root.parents[1]
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from study_location import define_location
definition = define_location(location_root / "config.yaml")
runtime_config = definition.config
config = runtime_config
grid_config = runtime_config
data_sources = runtime_config
sfincs_config = runtime_config
wflow_domain_review_required = True
runtime_config["wflow"]["domain_set"]["review_required"] = wflow_domain_review_required
wflow_config = {"wflow": runtime_config["wflow"]}

def resolve_location_path(relative_path):
    return location_root / relative_path

def ensure_parent(relative_path):
    path = resolve_location_path(relative_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

pd.set_option("display.max_colwidth", 120)

from sfincs_runs.scenarios import stage_inland_coupled_scenarios


## Required Inputs


In [ ]:
from wflow_runs.notebook import exists_table
exists_table(location_root, {
    "probability catalog parquet": "data/event_catalog/catalog/probability_catalog.parquet",
    "probability catalog csv": "data/event_catalog/catalog/probability_catalog.csv",
    "resilience stress/training catalog": "data/event_catalog/catalog/resilience_stress_training_catalog.csv",
    "wflow replay set": "data/event_catalog/catalog/wflow_replay_set.parquet",
    "event manifest": "data/event_catalog/event_manifest.yaml",
    "wflow handoff manifest": wflow_config["wflow"]["handoff"]["manifest"],
    "sfincs base model": sfincs_config["paths"]["base_model_root"],
})


## Scenario Build Plan


In [ ]:
scenario_build = sfincs_config["scenario_build"]
pd.Series({
    "design_scenario": scenario_build["design_scenario"],
    "forcing_variable": scenario_build["forcing_variable"],
    "zsini_mode": scenario_build["zsini_mode"],
    "spinup_hours": scenario_build["timing"]["spinup_hours"],
    "drain_down_hours": scenario_build["timing"]["drain_down_hours"],
    "min_run_hours": scenario_build["timing"]["min_run_hours"],
    "max_run_hours": scenario_build["timing"]["max_run_hours"],
})


## Wflow Event Forcing


In [ ]:
pd.Series({
    "events_root": wflow_config["wflow"]["events_root"],
    "source_variable": wflow_config["wflow"]["handoff"]["source_variable"],
    "handoff_manifest": wflow_config["wflow"]["handoff"]["manifest"],
    "reference_time": wflow_config["wflow"]["event_forcing"]["event_reference_time"],
})


## SFINCS Scenario Folders


In [ ]:
build_scenarios = False
scenario_catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
stress_training_catalog_path = resolve_location_path("data/event_catalog/catalog/resilience_stress_training_catalog.csv")
probability_catalog_path = resolve_location_path("data/event_catalog/catalog/probability_catalog.csv")

if build_scenarios:
    scenario_report = stage_inland_coupled_scenarios(
        runtime_config,
        {"location_root": location_root},
        catalog_path=scenario_catalog_path,
        force=False,
    )
else:
    scenarios_root = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
    scenario_count = len(pd.read_csv(scenario_catalog_path)) if scenario_catalog_path.exists() else 0
    stress_training_count = len(pd.read_csv(stress_training_catalog_path)) if stress_training_catalog_path.exists() else 0
    scenario_report = pd.Series({
        "build_scenarios": build_scenarios,
        "scenario_catalog_csv": str(scenario_catalog_path),
        "scenario_count": scenario_count,
        "stress_training_catalog_csv": str(stress_training_catalog_path),
        "stress_training_count": stress_training_count,
        "probability_catalog_csv": str(probability_catalog_path),
        "target_root": str(scenarios_root),
        "status": "review gated",
    })

scenario_report


## Scenario Folder Audit


In [ ]:
exists_table(location_root, {
    "scenarios root": sfincs_config["paths"]["scenarios_root"],
    "run stage root": sfincs_config["paths"]["run_root"],
    "run outputs root": sfincs_config["paths"]["storage_root"],
    "stats root": sfincs_config["paths"]["stats_root"],
})


## Cluster Launch Plan


In [ ]:
run_settings = sfincs_config["scenario_run"]
pd.Series({
    "workers": run_settings["workers"],
    "sfincs_bin": run_settings["sfincs_bin"],
    "sfincs_bin_env": run_settings["sfincs_bin_env"],
    "run_root": sfincs_config["paths"]["run_root"],
    "storage_root": sfincs_config["paths"]["storage_root"],
})
